# 04 — Pretrain diagnostics

Inspect what the unsupervised foundation model learned on Cambridge.

Prereqs:
  - `rt-tokenise --bbox 0.10,52.18,0.16,52.22 --site uk --out webapp/tokens.geojson`
  - `rt-pretrain --geojson webapp/tokens.geojson --runs-dir runs/cambridge_v1 --epochs 300`
  - `rt-embed --geojson webapp/tokens.geojson --ckpt runs/cambridge_v1/best.pt --out-geojson webapp/embeddings.geojson`

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
EMB_GEOJSON = ROOT / 'webapp' / 'embeddings.geojson'
EMB_PARQUET = ROOT / 'webapp' / 'embeddings.parquet'
RUN_DIR = ROOT / 'runs' / 'cambridge_v1'

print('config:', json.loads((RUN_DIR / 'config.json').read_text()))

## Build the diagnostic table

One row per token with: UMAP coords, cluster id, highway class, posted/ML/rule misalignment, crash data.

In [ ]:
gj = json.loads(EMB_GEOJSON.read_text())
props = pd.DataFrame([f['properties'] for f in gj['features']])
print('shape:', props.shape)
props.head(3)[['name','highway','posted_speed_kph','safe_system_speed_kph','misalignment_kph','ml_misalignment_kph','cluster','umap_x','umap_y']]

## UMAP scatter coloured by highway class

If the encoder learned to discriminate road class from masked context, points of the same `highway` should cluster together in UMAP space — even though `highway` was **not** seen as a target during pretraining (it was only one of many features that got masked).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
top_hw = props.highway.value_counts().head(8).index.tolist()
palette = plt.cm.tab10(np.linspace(0, 1, len(top_hw)))
for color, hw in zip(palette, top_hw):
    sub = props[props.highway == hw]
    ax.scatter(sub.umap_x, sub.umap_y, s=4, alpha=0.5, color=color, label=f'{hw} (n={len(sub)})')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title('Embedding UMAP — coloured by OSM highway class')
ax.legend(markerscale=3, loc='best', fontsize=9, framealpha=0.95)
ax.grid(alpha=0.3); fig.tight_layout()

## UMAP scatter coloured by k-means cluster

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
n_clusters = props.cluster.nunique()
cluster_palette = plt.cm.tab20(np.linspace(0, 1, n_clusters))
for color, cid in zip(cluster_palette, sorted(props.cluster.unique())):
    sub = props[props.cluster == cid]
    ax.scatter(sub.umap_x, sub.umap_y, s=4, alpha=0.6, color=color, label=f'C{cid} (n={len(sub)})')
ax.set_xlabel('UMAP-1'); ax.set_ylabel('UMAP-2')
ax.set_title(f'Embedding UMAP — coloured by k-means cluster (k={n_clusters})')
ax.legend(markerscale=3, loc='best', fontsize=8, ncol=2, framealpha=0.95)
ax.grid(alpha=0.3); fig.tight_layout()

## Cluster vs highway composition

If the SSL objective worked, each cluster should be dominated by a single highway class — i.e. the model learned road-class semantics without `highway` ever being a training target.

In [ ]:
ctab = props.groupby(['cluster', 'highway']).size().unstack(fill_value=0)
# Normalise rows to percentages
row_pct = (ctab.T / ctab.sum(axis=1)).T * 100
row_pct = row_pct.round(0).astype(int)
row_pct.loc['__total_count__'] = ctab.sum(axis=0).astype(int)
row_pct

## ML misalignment — distribution + agreement with rule-based

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(props.ml_misalignment_kph, bins=30, color='#2b5d8b', alpha=0.85)
axes[0].set_yscale('log')
axes[0].set_xlabel('ML misalignment (km/h)'); axes[0].set_ylabel('Token count (log)')
axes[0].set_title('ML misalignment distribution (k-NN posted-speed mode)')
axes[0].grid(alpha=0.3)

axes[1].scatter(props.misalignment_kph, props.ml_misalignment_kph, s=4, alpha=0.3, color='#d97706')
axes[1].plot([0, 50], [0, 50], 'k--', alpha=0.4, label='y=x')
axes[1].set_xlabel('Rule misalignment (km/h)'); axes[1].set_ylabel('ML misalignment (km/h)')
axes[1].set_title('Independent signals: rule vs ML misalignment')
axes[1].legend(); axes[1].grid(alpha=0.3)
fig.tight_layout()

Segments where the rule engine and the unsupervised model **both** disagree with the posted limit are the high-confidence candidates for review — they show up in the upper-right corner.

In [ ]:
both = props[(props.misalignment_kph >= 15) & (props.ml_misalignment_kph >= 15)]
print(f'Both signals flag {len(both)} tokens (rule>=15 AND ml>=15)')
both.sort_values('ml_misalignment_kph', ascending=False).drop_duplicates('name').head(15)[
    ['name', 'highway', 'posted_speed_kph', 'safe_system_speed_kph',
     'misalignment_kph', 'ml_misalignment_kph', 'cluster', 'crash_count']
]

## Top ML-misalignment-only segments (rule was silent)

These are interesting: the rule engine had no opinion, but the foundation model thinks the posted limit doesn't match what geometrically-similar segments carry. Candidate findings the rule-based pipeline missed.

In [ ]:
ml_only = props[(props.misalignment_kph < 5) & (props.ml_misalignment_kph >= 10)]
print(f'ML-only flags: {len(ml_only)} tokens (rule<5 AND ml>=10)')
ml_only.sort_values('ml_misalignment_kph', ascending=False).drop_duplicates('name').head(15)[
    ['name', 'highway', 'posted_speed_kph', 'safe_system_speed_kph',
     'misalignment_kph', 'ml_misalignment_kph', 'cluster', 'crash_count']
]

## Nearest-neighbour exploration in embedding space

Pick a token, find its k=10 most similar segments by embedding cosine distance, and inspect what they have in common. This is the building block of the ML misalignment scorer.

In [ ]:
emb_df = pd.read_parquet(EMB_PARQUET)
emb = emb_df.filter(like='e').values
emb_norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)

# Pick the highest ML-misalignment token to inspect
anchor_idx = int(props.ml_misalignment_kph.idxmax())
anchor = props.iloc[anchor_idx]
print(f'Anchor: token_id={anchor.token_id}, {anchor.get("name")}, {anchor.highway}, '
      f'posted={anchor.posted_speed_kph} km/h, ml_mis={anchor.ml_misalignment_kph:.0f}')

sims = emb_norm @ emb_norm[anchor_idx]
sims[anchor_idx] = -1
top_k_idx = np.argpartition(-sims, kth=10)[:10]
top_k_idx = top_k_idx[np.argsort(-sims[top_k_idx])]

neighbours = props.iloc[top_k_idx].copy()
neighbours['cosine_sim'] = sims[top_k_idx].round(3)
neighbours[['name', 'highway', 'posted_speed_kph', 'misalignment_kph', 'cluster', 'cosine_sim']]